In [1]:
import os
import math
import time
import argparse
import numpy as np
from tqdm import tqdm
from numpy.testing._private.utils import print_assert_equal

import torch
from torch import optim
from torch.utils.data import dataset
from numpy.core.fromnumeric import shape

from torchsummary import summary

import utils.loss
import utils.utils
import utils.datasets
import model.detector

c:\Users\Michael\anaconda3\envs\Yolo_new\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
cfg = utils.utils.load_datafile("./data/coco.data")

print(cfg)

train_dataset = utils.datasets.TensorDataset(
    cfg["train"], cfg["width"], cfg["height"], imgaug=True
)
val_dataset = utils.datasets.TensorDataset(
    cfg["val"], cfg["width"], cfg["height"], imgaug=False
)

batch_size = int(cfg["batch_size"] / cfg["subdivisions"])
nw = min([os.cpu_count(), batch_size if batch_size > 1 else 0, 8])

train_dataloader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=utils.datasets.collate_fn,
    num_workers=nw,
    pin_memory=True,
    drop_last=True,
    persistent_workers=True,
)

val_dataloader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=utils.datasets.collate_fn,
    num_workers=nw,
    pin_memory=True,
    drop_last=False,
    persistent_workers=True,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

load_param = False
premodel_path = cfg["pre_weights"]
if premodel_path != None and os.path.exists(premodel_path):
    load_param = True

model = model.detector.Detector(cfg["classes"], cfg["anchor_num"], load_param).to(
    device
)
summary(model, input_size=(3, cfg["height"], cfg["width"]))

if load_param == True:
    model.load_state_dict(
        torch.load(premodel_path, map_location=device), strict=False
    )
    print("Load finefune model param: %s" % premodel_path)
else:
    print("Initialize weights: model/backbone/backbone.pth")

optimizer = optim.SGD(
    params=model.parameters(),
    lr=cfg["learning_rate"],
    momentum=0.949,
    weight_decay=0.0005,
)

scheduler = optim.lr_scheduler.MultiStepLR(
    optimizer, milestones=cfg["steps"], gamma=0.1
)

print("Starting training for %g epochs..." % cfg["epochs"])

batch_num = 0
for epoch in range(cfg["epochs"]):
    model.train()
    pbar = tqdm(train_dataloader)

    for imgs, targets in pbar:
        imgs = imgs.to(device).float() / 255.0
        targets = targets.to(device)

        preds = model(imgs)

        iou_loss, obj_loss, cls_loss, total_loss = utils.loss.compute_loss(
            preds, targets, cfg, device
        )

        total_loss.backward()

        for g in optimizer.param_groups:
            warmup_num = 5 * len(train_dataloader)
            if batch_num <= warmup_num:
                scale = math.pow(batch_num / warmup_num, 4)
                g["lr"] = cfg["learning_rate"] * scale

            lr = g["lr"]

        if batch_num % cfg["subdivisions"] == 0:
            optimizer.step()
            optimizer.zero_grad()

        info = "Epoch:%d LR:%f CIou:%f Obj:%f Cls:%f Total:%f" % (
            epoch,
            lr,
            iou_loss,
            obj_loss,
            cls_loss,
            total_loss,
        )
        pbar.set_description(info)

        batch_num += 1

    if epoch % 10 == 0 and epoch > 0:
        model.eval()

        print("computer mAP...")
        _, _, AP, _ = utils.utils.evaluation(val_dataloader, cfg, model, device)
        print("computer PR...")
        precision, recall, _, f1 = utils.utils.evaluation(
            val_dataloader, cfg, model, device, 0.3
        )
        print("Precision:%f Recall:%f AP:%f F1:%f" % (precision, recall, AP, f1))

        torch.save(
            model.state_dict(),
            "weights/%s-%d-epoch-%fap-model.pth" % (cfg["model_name"], epoch, AP),
        )

    scheduler.step()


{'model_name': 'coco', 'epochs': 300, 'steps': [150.0, 250.0], 'batch_size': 128, 'subdivisions': 1, 'learning_rate': 0.001, 'pre_weights': 'None', 'classes': 80, 'width': 352, 'height': 352, 'anchor_num': 3, 'anchors': [12.64, 19.39, 37.88, 51.48, 55.71, 138.31, 126.91, 78.23, 131.57, 214.55, 279.92, 258.87], 'val': '/media/qiuqiu/D/coco/val2017.txt', 'train': '/media/qiuqiu/D/coco/train2017.txt', 'names': './data/coco.names'}


AssertionError: /media/qiuqiu/D/coco/train2017.txt文件路径错误或不存在

In [16]:
import os
import cv2
import time

import torch
import model.detector
import utils.utils


In [17]:
data_dir = "data/coco.data"
weights_dir = "modelzoo/coco2017-0.241078ap-model.pth"
img_dir = "img/000139.jpg"
cfg = utils.utils.load_datafile(data_dir)
assert os.path.exists(weights_dir), "请指定正确的模型路径"
assert os.path.exists(img_dir), "请指定正确的测试图像路径"

In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.detector.Detector(cfg["classes"], cfg["anchor_num"], True).to(device)
model.load_state_dict(torch.load(weights_dir, map_location=device))

#sets the module in eval node
model.eval()

#数据预处理
ori_img = cv2.imread(img_dir)
res_img = cv2.resize(ori_img, (cfg["width"], cfg["height"]), interpolation = cv2.INTER_LINEAR) 
img = res_img.reshape(1, cfg["height"], cfg["width"], 3)
img = torch.from_numpy(img.transpose(0,3, 1, 2))
img = img.to(device).float() / 255.0

#模型推理
start = time.perf_counter()
preds = model(img)
end = time.perf_counter()
time = (end - start) * 1000.
print("forward time:%fms"%time)

#特征图后处理
output = utils.utils.handel_preds(preds, cfg, device)
output_boxes = utils.utils.non_max_suppression(output, conf_thres = 0.3, iou_thres = 0.4)

#加载label names
LABEL_NAMES = []
with open(cfg["names"], 'r') as f:
    for line in f.readlines():
        LABEL_NAMES.append(line.strip())

h, w, _ = ori_img.shape
scale_h, scale_w = h / cfg["height"], w / cfg["width"]

#绘制预测框
for box in output_boxes[0]:
    box = box.tolist()
    
    obj_score = box[4]
    category = LABEL_NAMES[int(box[5])]

    x1, y1 = int(box[0] * scale_w), int(box[1] * scale_h)
    x2, y2 = int(box[2] * scale_w), int(box[3] * scale_h)

    cv2.rectangle(ori_img, (x1, y1), (x2, y2), (255, 255, 0), 2)
    cv2.putText(ori_img, '%.2f' % obj_score, (x1, y1 - 5), 0, 0.7, (0, 255, 0), 2)	
    cv2.putText(ori_img, category, (x1, y1 - 25), 0, 0.7, (0, 255, 0), 2)

cv2.imwrite("test_result.png", ori_img)


load param...
forward time:14.497800ms


-1

In [19]:
cv2.imshow("test",ori_img)
cv2.waitKey(0)

-1

In [2]:
import sys
sys.executable


'c:\\Program Files (x86)\\Microsoft Visual Studio\\Shared\\Python39_64\\python.exe'